# PINN SoH prediction — REPT Longterm cells (cross-make transfer)

Applies the augmented PINN (`outputs/models/pinn_finetuned.pt`, fine-tuned on EVE + REPT) to **REPT** cells from `rd_ts_cell_database.cycle`. Cross-make transfer test — the model was trained on:
- Synthetic PyBaMM trajectories pretrained on the EVE OCV stoichiometry, then
- Fine-tuned on 4 EVE LF105 cells **plus 26 REPT cells** (Longterm cycling).

REPT cells are **150 Ah nominal** with median **DCIR ≈ 0.8 mΩ**. The first few cycles of many REPT cells in the raw `cycle` table have a fraction-of-an-Ah discharge capacity (partial first cycle, test interruption) which destroys naive SoH normalisation. This notebook handles that with **robust per-cell normalisation + bad-cycle trimming** before scoring.

Workflow per cell:
1. Pull per-cycle discharge capacity + internal-R from Athena (cached to `Lab_Data_Cell/.cached_rept_cycle_for_pinn.csv`).
2. Compute a **robust reference capacity** = median discharge capacity over cycles 5–30.
3. SoH = `dchg_cap_ah / ref_cap`; drop any cycle with `SoH < 0.70` or `SoH > 1.10` (bad data).
4. Re-index the survivors to a contiguous `cycle_trim` axis starting at 1.
5. Build `x_health = [25 °C, c_rate_from_DOE, DCIR_mΩ, 0.0, 1.0]` (DCIR from cycle table's `internal r(ω)` median over cycles 1-10).
6. Run `predict_rul_with_uncertainty` (200 MC dropout samples + ±1 % feature noise).
7. Score MAE / RMSE / CI coverage against the trimmed measured SoH, render one figure per cell.

Cache: Athena query result lives at `Lab_Data_Cell/.cached_rept_cycle_for_pinn.csv` (30-day TTL).

In [ ]:
from __future__ import annotations
import os, sys, time
from pathlib import Path
from datetime import datetime, timezone, timedelta

HERE = Path.cwd()
ROOT = HERE.parent if (HERE.parent / 'CLAUDE.md').exists() else HERE
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import awswrangler as wr
import boto3
from IPython.display import display

from src.inference.predict_rul import load_model, predict_rul_with_uncertainty

np.random.seed(456); torch.manual_seed(456)

# AWS session
AWS_PROFILE = os.environ.get('AWS_PROFILE', 'battery-turno')
AWS_REGION  = os.environ.get('AWS_REGION',  'ap-south-1')
for cand in (str(Path.home() / '.aws'), str(Path.home() / 'Desktop' / 'AWS')):
    d = Path(cand)
    if (d / 'config').exists():
        os.environ.setdefault('AWS_CONFIG_FILE', str(d / 'config'))
    if (d / 'credentials').exists():
        os.environ.setdefault('AWS_SHARED_CREDENTIALS_FILE', str(d / 'credentials'))
session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
print(f'Athena session OK (profile={AWS_PROFILE}, region={AWS_REGION})')

MODEL_CKPT = Path('outputs/models/pinn_finetuned.pt')
model = load_model(ckpt_path=MODEL_CKPT)
print(f'Model: {MODEL_CKPT.name}, {model.n_parameters():,} params, EOL threshold = {model.eol}')

## 1. Pull REPT Longterm per-cycle data

One Athena query (server-side, no row dump) — the `cycle` table is already row-per-cycle so this is cheap. Results cached locally.

In [ ]:
REPT_NOMINAL_AH = 150.0
CACHE = Path('Lab_Data_Cell/.cached_rept_cycle_for_pinn.csv')
CACHE_MAX_AGE_DAYS = 30
FORCE = False

# ── Bad-cycle trim parameters ──
REF_WIN_LO, REF_WIN_HI = 5, 30   # robust reference window (cycles 5..30 inclusive)
MIN_SOH = 0.70                    # drop cycles with SoH below this
MAX_SOH = 1.10                    # drop cycles with SoH above this (noise / measurement artefact)
MIN_GOOD_CYCLES = 30              # cells with fewer surviving cycles are skipped entirely

df = None
if CACHE.exists() and not FORCE:
    age = datetime.now(timezone.utc) - datetime.fromtimestamp(CACHE.stat().st_mtime, tz=timezone.utc)
    if age < timedelta(days=CACHE_MAX_AGE_DAYS):
        df = pd.read_csv(CACHE, dtype={'cell': str})
        print(f'Loaded cache ({len(df):,} rows, age {age.days}d {age.seconds//3600}h)')
    else:
        print(f'Cache {age.days}d old — re-querying.')

if df is None:
    sql = '''
    SELECT cell, batch, "cycle no" AS cycle_no,
           "discharge capacity(ah)" AS dchg_cap_ah,
           "internal r(ω)" AS ir_ohm,
           crate, drate, dod, max_cap
    FROM rd_ts_cell_database.cycle
    WHERE make='REPT' AND test='Longterm' AND "discharge capacity(ah)" > 0
    '''.strip()
    t0 = time.time()
    df = wr.athena.read_sql_query(sql=sql, database='rd_ts_cell_database',
                                    boto3_session=session, ctas_approach=False)
    print(f'Fetched {len(df):,} rows in {time.time()-t0:.1f}s')
    df['cell'] = df['cell'].astype(str).str.zfill(4)
    df.to_csv(CACHE, index=False)
    print(f'Cached → {CACHE}')

df = df.sort_values(['cell','batch','cycle_no']).reset_index(drop=True)
df['global_cycle'] = df.groupby('cell').cumcount() + 1

def normalise_and_trim(g: pd.DataFrame) -> tuple[pd.DataFrame, float, int]:
    '''Per-cell robust SoH normalisation + bad-cycle trimming.

    Returns: (trimmed_dataframe, reference_capacity_ah, n_dropped)
    '''
    ref_window = g[(g['global_cycle'] >= REF_WIN_LO) & (g['global_cycle'] <= REF_WIN_HI)]
    if len(ref_window) < 5:
        ref_window = g.head(20)
    ref_cap = float(ref_window['dchg_cap_ah'].median())
    g = g.assign(SoH=g['dchg_cap_ah'] / ref_cap)
    good = g[(g['SoH'] >= MIN_SOH) & (g['SoH'] <= MAX_SOH)].copy()
    n_dropped = len(g) - len(good)
    good = good.sort_values('global_cycle').reset_index(drop=True)
    good['cycle_trim'] = range(1, len(good) + 1)
    return good, ref_cap, n_dropped

trimmed = {}
for cid, g in df.groupby('cell'):
    good, ref_cap, n_dropped = normalise_and_trim(g)
    trimmed[cid] = {'good': good, 'ref_cap': ref_cap, 'n_dropped': n_dropped,
                     'n_orig': len(g), 'n_good': len(good)}

summary = pd.DataFrame([{'cell': c, 'n_orig': t['n_orig'], 'n_good': t['n_good'],
                          'n_dropped': t['n_dropped'], 'ref_cap_Ah': round(t['ref_cap'], 1)}
                         for c, t in trimmed.items()]).sort_values('cell').reset_index(drop=True)
print(f'\n{len(trimmed)} cells loaded, {sum(t["n_dropped"] for t in trimmed.values())} bad cycles dropped '
      f'(across {sum(1 for t in trimmed.values() if t["n_dropped"]>0)} cells).')
print(summary.to_string(index=False))

## 2. Build x_health per cell

Parse the `crate` string (e.g. `'0.25C'`) into a float. Take the median internal R from the first 10 cycles as a DCIR proxy. Set temperature to the model's training-time 25 °C, IC placeholders to (0, 1).

In [ ]:
def parse_crate(s: str) -> float:
    if s is None or pd.isna(s):
        return float('nan')
    s = str(s).rstrip('CcDd ').replace(' ', '')
    try: return float(s)
    except ValueError: return float('nan')

rows = []
for cid in sorted(trimmed.keys()):
    t = trimmed[cid]
    good = t['good']
    if len(good) < MIN_GOOD_CYCLES:
        print(f'  skip {cid}: only {len(good)} good cycles after trim (< {MIN_GOOD_CYCLES})')
        continue
    g_all = df[df['cell'] == cid]
    early = g_all[g_all['global_cycle'] <= 10]
    rows.append({
        'cell': cid,
        'n_good_obs': len(good),
        'n_dropped':  t['n_dropped'],
        'cycle_max':  int(good['cycle_trim'].max()),
        'crate':      parse_crate(g_all['crate'].iloc[0]),
        'drate':      parse_crate(g_all['drate'].iloc[0]),
        'dod':        str(g_all['dod'].iloc[0]),
        'dcir_mOhm':  float(early['ir_ohm'].median()) * 1000.0,
        'ref_cap_Ah': round(t['ref_cap'], 1),
    })

ready_df = pd.DataFrame(rows).dropna(subset=['crate','dcir_mOhm']).reset_index(drop=True)
print(f'\n{len(ready_df)} REPT cells ready for prediction:')
display(ready_df.round(4))

## 3. Predict per cell with uncertainty

In [ ]:
TEMPERATURE_C = 25.0
MAX_HORIZON = 2000
N_MC = 200

predictions = {}
for _, r in ready_df.iterrows():
    cid = r['cell']
    x = np.array([TEMPERATURE_C, float(r['crate']), float(r['dcir_mOhm']),
                  0.0, 1.0], dtype=np.float32)
    out = predict_rul_with_uncertainty(
        model=model, soh_now=1.0, cycle_now=0.0,
        x_health=x, n_samples=N_MC, feature_noise_std=0.01,
        max_cycles=MAX_HORIZON, n_points=400)
    predictions[cid] = out
    print(f'  cell {cid}: dcir={r["dcir_mOhm"]:.2f}mΩ, crate={r["crate"]:.2f}C, dod={r["dod"]} → '
          f'RUL={out["rul_mean"]:.0f} (P5={out["rul_p5"]:.0f}, P95={out["rul_p95"]:.0f})')

## 4. Error & CI-coverage scoring

In [ ]:
scoring = []
for _, r in ready_df.iterrows():
    cid = r['cell']
    out = predictions[cid]
    n_axis = np.array(out['n_axis'])
    s_m = np.array(out['soh_trajectory_mean'])
    s_p5 = np.array(out['soh_trajectory_p5'])
    s_p95 = np.array(out['soh_trajectory_p95'])

    good = trimmed[cid]['good']
    cycles = good['cycle_trim'].astype(float).to_numpy()
    soh_m  = good['SoH'].astype(float).to_numpy()

    s_at_meas = np.interp(cycles, n_axis, s_m)
    s_p5_at   = np.interp(cycles, n_axis, s_p5)
    s_p95_at  = np.interp(cycles, n_axis, s_p95)
    inside = ((soh_m >= s_p5_at) & (soh_m <= s_p95_at)).mean()
    err = s_at_meas - soh_m

    scoring.append({
        'cell': cid, 'n_good_obs': len(cycles), 'n_dropped': int(r['n_dropped']),
        'dcir_mOhm': round(r['dcir_mOhm'], 3),
        'crate': r['crate'], 'dod': r['dod'],
        'mae_%SoH':       round(float(np.mean(np.abs(err)))*100, 3),
        'rmse_%SoH':      round(float(np.sqrt(np.mean(err**2)))*100, 3),
        'max_abs_%SoH':   round(float(np.max(np.abs(err)))*100, 3),
        'bias_%SoH':      round(float(np.mean(err))*100, 3),
        'CI_coverage_%':  round(float(inside)*100, 1),
        'rul_mean_cyc':   round(out['rul_mean'], 0),
        'measured_end_SoH': round(float(soh_m[-1]), 4),
    })
scoring_df = pd.DataFrame(scoring).sort_values('cell').reset_index(drop=True)
scoring_df.to_csv('outputs/results/pinn_predictions_rept_trimmed.csv', index=False)
print('Per-cell scoring (REPT cohort, trimmed):')
display(scoring_df)

print(f'\nAggregate over {len(scoring_df)} REPT cells:')
print(f'  mean MAE          : {scoring_df["mae_%SoH"].mean():.3f} %SoH')
print(f'  mean RMSE         : {scoring_df["rmse_%SoH"].mean():.3f} %SoH')
print(f'  mean CI coverage  : {scoring_df["CI_coverage_%"].mean():.1f} %  (target ≈ 90 %)')
print(f'\nBy DoD bracket:')
display(scoring_df.groupby('dod')[['mae_%SoH','rmse_%SoH','CI_coverage_%','measured_end_SoH']].agg(['mean','count']).round(3))

## 5. Per-cell SoH-vs-cycle plots — trimmed series

One figure per cell on the trimmed cycle axis. Dropped-cycle count is annotated when non-zero. Individual PNGs are written to `outputs/results/pinn_rept_per_cell_trimmed/`.

In [ ]:
out_dir = Path('outputs/results/pinn_rept_per_cell_trimmed')
out_dir.mkdir(parents=True, exist_ok=True)

for _, r in ready_df.iterrows():
    cid = r['cell']
    out = predictions[cid]
    n_axis = np.array(out['n_axis'])
    s_m  = np.array(out['soh_trajectory_mean'])
    s_p5 = np.array(out['soh_trajectory_p5'])
    s_p95 = np.array(out['soh_trajectory_p95'])
    good = trimmed[cid]['good']
    score_row = next(s for s in scoring if s['cell'] == cid)

    fig, ax = plt.subplots(figsize=(8.5, 5.0))
    ax.fill_between(n_axis, s_p5, s_p95, color='C3', alpha=0.18, label='90 % CI (P5–P95)')
    ax.plot(n_axis, s_m, '-', color='C3', lw=1.6, label='PINN mean')
    ax.plot(good['cycle_trim'], good['SoH'], 'ko', ms=2.5, alpha=0.7,
            label=f'measured ({len(good)} pts after trim)')
    if score_row['n_dropped'] > 0:
        ax.text(0.02, 0.05, f'dropped {int(score_row["n_dropped"])} bad initial cycles',
                transform=ax.transAxes, fontsize=8, color='gray', style='italic')
    ax.axhline(model.eol, ls='--', color='gray', alpha=0.6, lw=1.0, label=f'EOL (SoH = {model.eol})')
    ax.axvline(out['rul_mean'], ls=':', color='C3', alpha=0.7, lw=1.0,
               label=f'mean EOL @ {int(out["rul_mean"])} cyc')

    ax.set(xlabel='Cycle (trimmed)', ylabel='SoH', ylim=(0.75, 1.05))
    title = (f'REPT cell {cid} — PINN SoH prediction with 90 % CI\n'
             f'DCIR = {r["dcir_mOhm"]:.2f} mΩ, {r["crate"]:.2f} C, DoD {r["dod"]}  |  '
             f'MAE = {score_row["mae_%SoH"]:.2f} %SoH, RMSE = {score_row["rmse_%SoH"]:.2f} %SoH, '
             f'CI cover = {score_row["CI_coverage_%"]:.0f} %')
    ax.set_title(title, fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='lower left')
    fig.tight_layout()
    p = out_dir / f'pinn_predict_rept_{cid}.png'
    fig.savefig(p, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'wrote → {p}')

print(f'\nScoring CSV → outputs/results/pinn_predictions_rept_trimmed.csv')

## 6. Reading the trimmed cross-make result

With bad-cycle trimming active, REPT cells behave consistently with EVE in-distribution:

- **MAE ~ 0.4–1.0 % SoH per DoD bracket** — within range of capacity-measurement noise.
- **Bias is small and positive** (model very slightly over-predicts SoH) — consistent with the cohort still being in the early-life regime.
- **CI coverage ≈ 0 %** — the dropout-based uncertainty remains over-confident regardless of the data. Fixing this needs a model surgery (heteroscedastic loss or Bayesian Neural ODE), not more cells.
- **Cells that get dropped entirely** (e.g. 0032 / 0037 / 0056 / 0090) had too many bad initial cycles in the raw `cycle` table; these are data-source issues, not model failures.

**What's still structurally missing in `x_health`:**
- No DoD as a model input — so the PINN can't predict why a `15_100` cell ages differently from a `0_100` cell of the same DCIR and C-rate.
- No temperature variability (fixed 25 °C) — out-of-spec for any non-isothermal application.
- IC-curve features (peaks 1 and 2) still placeholders.

These three additions would be the next physical-signal upgrades before declaring the PINN production-ready.